# Classificação — Otimização Bayesiana (Optuna)

**Objetivo:** ajustar os hiperparâmetros do modelo vencedor da comparação (`03_Comparacao_Modelos.ipynb`, aqui assumido XGBoost — ajuste `ESPACO_BUSCA`/`objective` se outro modelo tiver vencido) usando busca bayesiana (TPE) do Optuna, otimizando F1 — métrica mais informativa que accuracy para o alvo desbalanceado `com_vitima_fatal`.

A otimização usa validação cruzada apenas no conjunto de **treino**; o conjunto de teste só é usado uma vez, no final, para avaliar o modelo com os melhores hiperparâmetros encontrados.

## 1. Imports

In [ ]:
import sys
sys.path.append('..')

import joblib
import matplotlib.pyplot as plt
import optuna
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

from src.data import RANDOM_STATE, TEST_SIZE, carregar_dados, construir_preprocessador, separar_x_y

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Carregar dados e split (mesmo critério dos notebooks anteriores)

In [ ]:
df = carregar_dados()
X, y = separar_x_y(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

escala_pos = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

## 3. Função objetivo

In [ ]:
def objective(trial):
    parametros = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }

    pipeline = Pipeline([
        ('preprocessador', construir_preprocessador()),
        ('modelo', XGBClassifier(
            **parametros, scale_pos_weight=escala_pos, eval_metric='logloss',
            random_state=RANDOM_STATE, n_jobs=-1)),
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    return scores.mean()

## 4. Rodar o estudo de otimização

In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('Melhor F1 (CV):', round(study.best_value, 4))
print('Melhores hiperparâmetros:', study.best_params)

## 5. Histórico da otimização

In [ ]:
plot_optimization_history(study)
plt.savefig('../reports/optuna_historico.png', dpi=150, bbox_inches='tight')
plt.show()

plot_param_importances(study)
plt.savefig('../reports/optuna_importancia_parametros.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Modelo final: treino completo e avaliação no teste

In [ ]:
pipeline_final = Pipeline([
    ('preprocessador', construir_preprocessador()),
    ('modelo', XGBClassifier(
        **study.best_params, scale_pos_weight=escala_pos, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1)),
])
pipeline_final.fit(X_train, y_train)

pred = pipeline_final.predict(X_test)
proba = pipeline_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap='Blues', ax=eixos[0])
eixos[0].set_title('Matriz de confusão — XGBoost otimizado')
RocCurveDisplay.from_predictions(y_test, proba, ax=eixos[1])
eixos[1].set_title('Curva ROC')
plt.tight_layout()
plt.savefig('../reports/modelo_final_avaliacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Salvar modelo final

In [ ]:
joblib.dump(pipeline_final, '../models/modelo_final_optuna.joblib')

## 8. Conclusão
- Baseline (LogReg) → comparação com validação cruzada (LogReg, RandomForest, XGBoost) → otimização bayesiana do modelo vencedor.
- Métricas de referência: F1 e ROC-AUC (accuracy é enganosa neste dataset desbalanceado).
- Modelo final salvo em `../models/modelo_final_optuna.joblib`.